In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow_decision_forests as tfdf
import pandas as pd

https://www.tensorflow.org/decision_forests/api_docs/python/tfdf/keras/RandomForestModel

In [3]:
predictors = ["grid", "position", "pos_delta", "driver_code", "constructor_code", "circuit_code", "grid_rolling", "position_rolling", "pos_delta_rolling"]

In [4]:
data = pd.read_csv("./data/final_rolling.csv")

In [5]:
data.dropna(subset="position", inplace=True)

In [6]:
data["position"] = data["position"].astype("int")

In [7]:
training = data[data["year"] < 2022]
test = data[data["year"] >= 2022]

In [8]:
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(training[predictors], label="position")
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(test[predictors])

In [9]:
model = tfdf.keras.RandomForestModel(verbose=0)

model.fit(train_ds)

[INFO 24-08-26 13:11:27.3656 CEST kernel.cc:1233] Loading model from path /var/folders/hk/ffpx1y0s2cn133v15g3yks080000gn/T/tmp6oc10ns9/model/ with prefix a67fbc0b539645f0
[INFO 24-08-26 13:11:27.7762 CEST decision_forest.cc:734] Model loaded with 300 root(s), 317270 node(s), and 8 input feature(s).
[INFO 24-08-26 13:11:27.7762 CEST abstract_model.cc:1344] Engine "RandomForestGeneric" built
[INFO 24-08-26 13:11:27.7762 CEST kernel.cc:1061] Use fast generic engine


In [10]:
predictions = model.predict(test_ds)

2/2 [==============================] - 0s 2ms/step


### Prediction tables:
    Rows - prediction number
    Column - driverId

In [11]:
predictions_df = pd.DataFrame(predictions)

In [12]:
full_table = pd.merge(test, predictions_df, on=test.index)

full_table

,key_0,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,...,11,12,13,14,15,16,17,18,19,20
0,4153,1074,1.0,1,2022,2022-03-20,15:00:00,bahrain,leclerc,ferrari,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,4154,1074,3.0,2,2022,2022-03-20,15:00:00,bahrain,sainz,ferrari,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,4155,1074,5.0,3,2022,2022-03-20,15:00:00,bahrain,hamilton,mercedes,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,4156,1074,9.0,4,2022,2022-03-20,15:00:00,bahrain,russell,mercedes,...,0.010000,0.023333,0.003333,0.000000,0.006667,0.003333,0.000000,0.000000,0.000000,0.000000
4,4157,1074,7.0,5,2022,2022-03-20,15:00:00,bahrain,kevin_magnussen,haas,...,0.026667,0.023333,0.023333,0.040000,0.016667,0.006667,0.016667,0.006667,0.003333,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1013,5166,1134,20.0,16,2024,2024-07-28,15:00,spa,tsunoda,rb,...,0.006667,0.053333,0.020000,0.163333,0.373333,0.273333,0.056667,0.033333,0.003333,0.006667
1014,5167,1134,18.0,17,2024,2024-07-28,15:00,spa,sargeant,williams,...,0.000000,0.010000,0.006667,0.033333,0.023333,0.250000,0.480000,0.156667,0.036667,0.003333
1015,5168,1134,16.0,18,2024,2024-07-28,15:00,spa,hulkenberg,haas,...,0.006667,0.030000,0.036667,0.093333,0.086667,0.110000,0.196667,0.326666,0.033333,0.040000
1016,5169,1134,19.0,19,2024,2024-07-28,15:00,spa,zhou,sauber,...,0.000000,0.003333,0.010000,0.013333,0.006667,0.036667,0.103333,0.393333,0.340000,0.093333


In [13]:
evaluation = model.make_inspector().evaluation()

In [14]:
evaluation.accuracy

0.7972549963881531

In [37]:
preds = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

In [38]:
full_table["max_pred"] = full_table[preds].max(axis=1)

In [44]:
full_table[["raceId", "driverRef", "constructorRef", "circuitRef", "grid", 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]]

,raceId,driverRef,constructorRef,circuitRef,grid,1,2,3,4,5,6,7,8,9,10
0,1074,leclerc,ferrari,bahrain,1.0,0.769999,0.070000,0.086667,0.043333,0.020000,0.003333,0.003333,0.000000,0.000000,0.003333
1,1074,sainz,ferrari,bahrain,3.0,0.143333,0.643333,0.176667,0.016667,0.006667,0.000000,0.013333,0.000000,0.000000,0.000000
2,1074,hamilton,mercedes,bahrain,5.0,0.063333,0.273333,0.503333,0.113333,0.036667,0.006667,0.000000,0.003333,0.000000,0.000000
3,1074,russell,mercedes,bahrain,9.0,0.003333,0.040000,0.093333,0.306666,0.156667,0.090000,0.086667,0.066667,0.073333,0.036667
4,1074,kevin_magnussen,haas,bahrain,7.0,0.010000,0.010000,0.050000,0.063333,0.190000,0.233333,0.066667,0.090000,0.096667,0.026667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1013,1134,tsunoda,rb,spa,20.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003333,0.000000,0.006667
1014,1134,sargeant,williams,spa,18.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1015,1134,hulkenberg,haas,spa,16.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.006667,0.000000,0.003333,0.010000,0.020000
1016,1134,zhou,sauber,spa,19.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [40]:
inspector = model.make_inspector()

In [41]:
full_table

,key_0,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,...,12,13,14,15,16,17,18,19,20,max_pred
0,4153,1074,1.0,1,2022,2022-03-20,15:00:00,bahrain,leclerc,ferrari,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.769999
1,4154,1074,3.0,2,2022,2022-03-20,15:00:00,bahrain,sainz,ferrari,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.643333
2,4155,1074,5.0,3,2022,2022-03-20,15:00:00,bahrain,hamilton,mercedes,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.503333
3,4156,1074,9.0,4,2022,2022-03-20,15:00:00,bahrain,russell,mercedes,...,0.023333,0.003333,0.000000,0.006667,0.003333,0.000000,0.000000,0.000000,0.000000,0.306666
4,4157,1074,7.0,5,2022,2022-03-20,15:00:00,bahrain,kevin_magnussen,haas,...,0.023333,0.023333,0.040000,0.016667,0.006667,0.016667,0.006667,0.003333,0.000000,0.233333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1013,5166,1134,20.0,16,2024,2024-07-28,15:00,spa,tsunoda,rb,...,0.053333,0.020000,0.163333,0.373333,0.273333,0.056667,0.033333,0.003333,0.006667,0.373333
1014,5167,1134,18.0,17,2024,2024-07-28,15:00,spa,sargeant,williams,...,0.010000,0.006667,0.033333,0.023333,0.250000,0.480000,0.156667,0.036667,0.003333,0.480000
1015,5168,1134,16.0,18,2024,2024-07-28,15:00,spa,hulkenberg,haas,...,0.030000,0.036667,0.093333,0.086667,0.110000,0.196667,0.326666,0.033333,0.040000,0.326666
1016,5169,1134,19.0,19,2024,2024-07-28,15:00,spa,zhou,sauber,...,0.003333,0.010000,0.013333,0.006667,0.036667,0.103333,0.393333,0.340000,0.093333,0.393333


In [45]:
tree = inspector.extract_tree(tree_idx=0)

In [60]:
import dtreeviz as dt

In [61]:
features = [f.name for f in model.make_inspector().features()]

In [66]:
features

['circuit_code',
 'constructor_code',
 'driver_code',
 'grid',
 'grid_rolling',
 'pos_delta',
 'pos_delta_rolling',
 'position_rolling']

In [63]:
training.dropna(inplace=True)

/var/folders/hk/ffpx1y0s2cn133v15g3yks080000gn/T/ipykernel_16231/736322629.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [65]:
training

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta,grid_rolling,position_rolling,pos_delta_rolling
48,340,5.0,1,2010,2010-04-18,06:00:00,shanghai,button,mclaren,24,11,13,4.0,9.666667,5.333333,4.333333
49,340,6.0,2,2010,2010-04-18,06:00:00,shanghai,hamilton,mclaren,24,23,13,4.0,11.666667,5.000000,6.666667
50,340,4.0,3,2010,2010-04-18,06:00:00,shanghai,rosberg,mercedes,24,60,14,1.0,4.333333,4.333333,0.000000
51,340,3.0,4,2010,2010-04-18,06:00:00,shanghai,alonso,ferrari,24,3,5,-1.0,8.333333,6.000000,2.333333
52,340,8.0,5,2010,2010-04-18,06:00:00,shanghai,kubica,renault,24,34,18,3.0,8.000000,5.666667,2.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4148,1073,15.0,11,2021,2021-12-12,13:00:00,yas_marina,vettel,aston_martin,32,73,3,4.0,12.000000,9.333333,2.666667
4149,1073,10.0,12,2021,2021-12-12,13:00:00,yas_marina,ricciardo,mclaren,32,58,13,-2.0,8.000000,7.333333,0.666667
4150,1073,13.0,13,2021,2021-12-12,13:00:00,yas_marina,stroll,aston_martin,32,67,3,0.0,17.000000,12.333333,4.666667
4151,1073,19.0,14,2021,2021-12-12,13:00:00,yas_marina,mick_schumacher,haas,32,47,7,5.0,16.000000,17.666667,-1.666667


In [72]:
viz_model = dt.model(
    model=model,
    X_train=training[features],
    y_train=training["position"]-1,
    feature_names=features,
    target_name="position",
    tree_index=0
)

In [88]:
viz_model.view(depth_range_to_display=[0,5], scale=0.75, orientation="LR").save("./tree.svg")